## Middleware

Middleware provides a way to more tightly control what happens inside the agent.
Middleware is useful for the following :

- Tracking agent behaviour with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails and PII Detection.


Build in middleware
- Summarization
- Human in the feedback
- Model call limit 


In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-3.8-flash")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7', 'langchain-google-genai': '4.4.2'}}, profile={'name': 'Gemini 3.8 Flash', 'release_date': '2026-09-02', 'last_updated': '2026-09-02', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-3.8-flash', temperature=None, client=<google.genai.client.Client object at 0x000001DD411057F0>, default_metadata=(), model_kwargs={})

## Summarization Middleware

Automatically summarize conversation history when approching token limit
, preserving recent messages while compressing older context.
Summarization is usefull for the following :
- Long-running conversation that exceed context windows.
- Multi-turn dialogues with extensive history
- Applications where preserving full conversation context matters.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    middleware=SummarizationMiddleware(
        model="google_genai:gemini-3.8-flash",
        trigger=("fraction", 0.005),
        keep=("fraction", 0.002),
    )
)

config = {"configurable": {"thread_id": "test-1"}}


## Token counter

def count_token(messages):
    return sum(len(str(m.content) for m in messages))
    

_IncompleteInputError: incomplete input (628720403.py, line 17)

## Human in the Loop middleware

Pause agent execution for human apporval, editing , or rejection of tool calls before they execute. Human-in-the-loop is used for the following:

- High-stakes operations requiring human apporval (e.g database writes, financial transaction)
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where feedback guides the agent.


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str)-> str:
    """Mock funnction to read an email by id ID."""
    return f"""Email content for ID : {email_id}"""

def send_email_tool(receipient: str, subject: str, body: str)-> str:
    """Mock funnction to sejd an email to ID."""
    return f"""Email sent to {receipient} with subject `{subject}`"""
    

In [4]:
agent = create_agent(
    model="google_genai:gemini-3.8-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware= [
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]

)

In [6]:
from langchain_core.messages import HumanMessage
config = {"configurable": {"thread_id": "test-approve"}}

## step 1: request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?' ")]},
    config= config
)


result


{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='e4b7b7e7-adf4-45cb-90c7-efc1649f41de'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "body": "How are you?", "receipient": "john@example.com"}'}, '__gemini_function_call_thought_signatures__': {'call_544017': 'ErcECrQEAWkUfRMOwDNzGYXCMY3iuaZ46tTY60yDfSkTSLVUugmUdaNuepZv1n7YpWW6SDQsxN8WIoP4mKllC6hgL5cDA3ZC38vRXQLx9dSd3hP1VE4ls3xWSqVDSn0/kFhgLpgsjS2ZhCCWyzH2Ms6HaETDFHFgtK6VUy6U5BSCZp0QRoaQV7W3JwqldvEWcULOFiitrEvV774zXkJPtNWFkEkFDuZIihuK5XsThe1mcVCA7fy8/W3r5bEMppttetYdckkLnXLheKuftZohU7PIL7WRbT/yZBuqqzCyAxNaP465WFmyC714GI1wZaUlzsU3739b5Mp7rPOY1vx302wzAgvAQu4hujvSBM1gT76Yn8Vvg+BquG9Rl3QCBP5oh/aP6xTAl36ZEqdQM07sSnqW5wIJmYvEOyLsf3cqzoA39kQOxqu1PsZl4gnSF4YL1kyk4B8raoQIJ+WQBCSpqQ5fXhKcRm4OAnQvNhbdTYqW2T48MhWMNtR+QAUDmvhzljz6nZbQZYbxg2ASGxYD0LTt26gc0Q0eidCxn

In [7]:

from langgraph.types import Command

## Step 2

if "__interrupt__" in result:
    print("--Pause ! Approving ... ")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f" Result {result['messages'][-1].content} ")
    

--Pause ! Approving ... 
 Result [{'type': 'text', 'text': 'The email to john@example.com with the subject "Hello" and body "How are you?" has been sent successfully.', 'extras': {'signature': 'EqcBCqQBAWkUfRN+Mx8uiL8HCIYGy1kd/EgkNda+iP4I5SAjP7kCtGPEs+F3RmksHe8j1AW6JQiFpzzZ4fGEmjuSNTI/pTijEstdGNijJx1ea9qascdFWQzYV+T14hnkC+dUh6MPkmew0Wa34SIFQNRMd1ykzoAQn8BGQFsGMwr/t9omVuvrWCwD7B89ZgsiyOq++vN0CDkGIxy1vVkC4Yb8BDNbhnfOrTE='}}] 
